In [18]:
# Imports and config
import cv2
import numpy as np
from skimage.segmentation import slic
from skimage.feature import graycomatrix, graycoprops
from pathlib import Path
import json

# Dataset paths
BASELINE_DIR = Path("data/normal_sea/baseline_split")     # used to build "what is normal water"
NEGATIVE_DIR = Path("data/normal_sea/heldout_split")       # held-out normal sea - never touched during baseline build
PRAWN_DIR    = Path("data/prawns")               # only the 42 deduplicated prawn images

IMG_SIZE = (512, 512)  # fixed resize since altitude/GSD isn't recoverable from prawn images

In [2]:
# Pre-Processing Functions
def load_and_resize(path, size=IMG_SIZE):
    img = cv2.imread(str(path))
    img = cv2.resize(img, size)
    return img

def suppress_glint(img_bgr, thresh=220, dilate_iter=2):
    """Masks out bright sun-glint pixels on the L channel."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = lab[:, :, 0]
    glint_mask = (L > thresh).astype(np.uint8)
    glint_mask = cv2.dilate(glint_mask, np.ones((5, 5), np.uint8), iterations=dilate_iter)
    return glint_mask  # 1 = glint, exclude from analysis

def to_lab(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

def preprocess_image(path):
    """Full chain: load -> resize -> LAB -> glint mask."""
    img = load_and_resize(path)
    lab = to_lab(img)
    glint_mask = suppress_glint(img)
    return img, lab, glint_mask

In [3]:
# SLIC + GLCM feature extraction
def extract_superpixel_features(img_lab, glint_mask, n_segments=200):
    """
    Runs SLIC on the L channel, then computes GLCM texture stats
    per superpixel — keeping true 2D spatial structure (this is the
    fix vs. the inherited flattened-1D bug).
    """
    L_channel = img_lab[:, :, 0]
    segments = slic(img_lab, n_segments=n_segments, compactness=10, start_label=1)

    features = []
    for seg_id in np.unique(segments):
        mask = segments == seg_id
        if glint_mask[mask].mean() > 0.5:
            continue  # skip glint-dominated superpixels entirely

        ys, xs = np.where(mask)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        patch = L_channel[y0:y1, x0:x1]  # real 2D patch, not flattened

        if patch.shape[0] < 2 or patch.shape[1] < 2:
            continue

        glcm = graycomatrix(patch, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
        contrast = graycoprops(glcm, 'contrast')[0, 0]
        homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
        energy = graycoprops(glcm, 'energy')[0, 0]

        features.append({
            "seg_id": int(seg_id),
            "bbox": (int(x0), int(y0), int(x1), int(y1)),
            "contrast": float(contrast),
            "homogeneity": float(homogeneity),
            "energy": float(energy),
        })
    return segments, features

In [4]:
# Normal Water Baseline
baseline_stats = {"contrast": [], "homogeneity": [], "energy": []}

for path in BASELINE_DIR.glob("*.jpg"):
    img, lab, glint_mask = preprocess_image(path)
    _, feats = extract_superpixel_features(lab, glint_mask)
    for f in feats:
        baseline_stats["contrast"].append(f["contrast"])
        baseline_stats["homogeneity"].append(f["homogeneity"])
        baseline_stats["energy"].append(f["energy"])

baseline_ref = {k: (np.mean(v), np.std(v)) for k, v in baseline_stats.items()}
print("Baseline (mean, std):", baseline_ref)

Baseline (mean, std): {'contrast': (np.float64(15.071422244605001), np.float64(45.56627301106702)), 'homogeneity': (np.float64(0.45867243820965276), np.float64(0.17072387327772548)), 'energy': (np.float64(0.09589421469939916), np.float64(0.08634727940393062))}


In [31]:
# Deviation scoring + detector runner
def score_region_deviation(feat, baseline_ref, z_thresh=2.5):
    z_scores = {
        k: abs(feat[k] - baseline_ref[k][0]) / (baseline_ref[k][1] + 1e-6)
        for k in ["contrast", "homogeneity", "energy"]
    }
    max_z = max(z_scores.values())
    return max_z > z_thresh, max_z

def run_detector(img_dir, baseline_ref, min_component_size=3):
    results = {}
    for path in Path(img_dir).glob("*.jpg"):
        img, lab, glint_mask = preprocess_image(path)
        segments, feats = extract_superpixel_features(lab, glint_mask)
        flagged = [f for f in feats if score_region_deviation(f, baseline_ref)[0]]
        # Only count as a detection if enough flagged superpixels form a real signal
        is_detected = len(flagged) >= min_component_size
        results[path.name] = {"n_flagged": len(flagged), "is_detected": is_detected, "flagged_regions": flagged}
    return results

In [33]:
# Run on negative controls
negative_results = run_detector(NEGATIVE_DIR, baseline_ref)
false_positive_rate = np.mean([1 if r["is_detected"] else 0 for r in negative_results.values()])
print(f"False-positive rate on plume-free water: {false_positive_rate:.2%}")

False-positive rate on plume-free water: 23.41%


In [22]:
# Run on Prawn Positives
positive_results = run_detector(PRAWN_DIR, baseline_ref)
prawn_detection_rate = np.mean([1 if r["n_flagged"] > 0 else 0 for r in positive_results.values()])
print(f"Detection rate on prawn plumes: {prawn_detection_rate:.2%}")

Detection rate on prawn plumes: 98.78%


In [1]:
summary = {
    "false_positive_rate": false_positive_rate,
    "prawn_detection_rate": prawn_detection_rate,
    "n_negative_images": len(negative_results),
    "n_positive_images": len(positive_results),
}
print(json.dumps(summary, indent=2))

with open("stage1_scored_baseline.json", "w") as f:
    json.dump(summary, f, indent=2)

NameError: name 'false_positive_rate' is not defined